# MLflow - Registro e Gestao de Modelos - Credit Risk Intelligence Platform

Este notebook integra MLflow ao pipeline de Machine Learning, registrando experimentos de forma reprodutivel e auditavel. Reutiliza os resultados reais dos notebooks `16_gold_training` e `17_evaluation`, registra cada modelo no MLflow com parametros, metricas, tags e artefatos, e os versiona no Model Registry do Unity Catalog.

**Dataset**: `credit_risk.gold.ml_train` (307.511 registros, 229 features)
**Modelos**: Dummy, Logistic Regression, Random Forest, XGBoost, LightGBM
**Importante**: NAO recalcula metricas, NAO faz tuning, NAO usa ml_test para avaliacao.

In [0]:
# ============================================================
# INSTALACAO DE BIBLIOTECAS (XGBoost e LightGBM)
# Necessario para recriar os modelos com a mesma configuracao
# ============================================================
%pip install xgboost lightgbm -q

In [0]:
# ============================================================
# SECAO 1 - CONFIGURACAO INICIAL
# ============================================================
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

import pandas as pd
import numpy as np
import time
import uuid
import warnings
import os
import tempfile
from datetime import datetime

from pyspark.sql import functions as F

# Imports de modelos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Tentar importar XGBoost e LightGBM
xgb_available = False
lgb_available = False

try:
    from xgboost import XGBClassifier
    xgb_available = True
    print("XGBoost disponivel")
except ImportError:
    print("WARNING: XGBoost nao disponivel")

try:
    from lightgbm import LGBMClassifier
    lgb_available = True
    print("LightGBM disponivel")
except ImportError:
    print("WARNING: LightGBM nao disponivel")

# Configuracoes globais
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Configuracoes do projeto
EXPERIMENT_NAME = "/Shared/Credit_Risk_Intelligence"
TRAIN_TABLE = "credit_risk.gold.ml_train"
EVALUATION_TABLE = "credit_risk.analytics.evaluation_results"
TRAINING_RESULTS_TABLE = "credit_risk.analytics.model_training_results"
THRESHOLD_TABLE = "credit_risk.analytics.threshold_analysis"
FEATURE_IMPORTANCE_TABLE = "credit_risk.analytics.feature_importance_training"
AUDIT_TABLE = "credit_risk.analytics.audit_evaluation"

# Catalog/schema para Model Registry no Unity Catalog
REGISTRY_CATALOG = "credit_risk"
REGISTRY_SCHEMA = "analytics"

# ID de execucao unico para auditoria
EXECUTION_ID = str(uuid.uuid4())
START_TIME = time.time()
START_TIMESTAMP = datetime.now()

# Detectar versao do MLflow para compatibilidade de API
# MLflow 3.x usa name=, MLflow 2.x usa artifact_path=
mlflow_version = mlflow.__version__
mlflow_major = int(mlflow_version.split('.')[0])
LOG_MODEL_KWARG = "name" if mlflow_major >= 3 else "artifact_path"

print(f"\nMLflow version: {mlflow_version} (major: {mlflow_major})")
print(f"log_model parameter: {LOG_MODEL_KWARG}=")

print(f"\n{'='*60}")
print(f"18_MLFLOW - CREDIT RISK INTELLIGENCE PLATFORM")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Start Time: {START_TIMESTAMP}")
print(f"{'='*60}")

In [0]:
# ============================================================
# SECAO 2 - CONFIGURACAO DO EXPERIMENTO MLFLOW
# ============================================================
print("=" * 60)
print("SECAO 2 - Configuracao do Experimento MLflow")
print("=" * 60)

# Criar ou selecionar o experimento
mlflow.set_experiment(EXPERIMENT_NAME)

# Obter informacoes do experimento
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
tracking_uri = mlflow.get_tracking_uri()

print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Tracking URI: {tracking_uri}")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Timestamp: {START_TIMESTAMP}")

# Configurar registry URI para Unity Catalog
mlflow.set_registry_uri("databricks-uc")
print(f"Registry URI: databricks-uc")

# Verificar se MLflow esta configurado
if experiment is None:
    print("ERRO: Falha ao criar/selecionar experimento")
else:
    print("MLflow configurado com sucesso!")

In [0]:
# ============================================================
# SECAO 3 - CARREGAR RESULTADOS DOS NOTEBOOKS ANTERIORES
# ============================================================
print("=" * 60)
print("SECAO 3 - Carregando resultados dos notebooks 16 e 17")
print("=" * 60)

# Carregar tabela de resultados de treinamento (notebook 16)
df_training_results = spark.table(TRAINING_RESULTS_TABLE).toPandas()
print(f"model_training_results: {len(df_training_results)} registros")
print(df_training_results[['model_name', 'roc_auc', 'pr_auc', 'recall', 'precision', 'f1']].to_string(index=False))

# Carregar tabela de resultados de avaliacao (notebook 17)
df_eval_results = spark.table(EVALUATION_TABLE).toPandas()
print(f"\nevaluation_results: {len(df_eval_results)} registros")
print(df_eval_results[['MODEL', 'ROC_AUC', 'PR_AUC', 'RECALL', 'PRECISION', 'F1', 'BRIER_SCORE']].to_string(index=False))

# Carregar analise de threshold
df_threshold = spark.table(THRESHOLD_TABLE).toPandas()
print(f"\nthreshold_analysis: {len(df_threshold)} registros")

# Carregar feature importance
df_feature_importance = spark.table(FEATURE_IMPORTANCE_TABLE).toPandas()
print(f"feature_importance_training: {len(df_feature_importance)} registros")
print(f"Modelos com feature importance: {df_feature_importance['model_name'].unique().tolist()}")

# Validacao: verificar se as metricas existem
required_models = ['Dummy (most_frequent)', 'Dummy (stratified)', 'Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']
print(f"\n--- Validacao de Metricas ---")
for model in required_models:
    in_training = model in df_training_results['model_name'].values
    in_eval = model in df_eval_results['MODEL'].values
    print(f"  {model}: training={'SIM' if in_training else 'NAO'} | evaluation={'SIM' if in_eval else 'NAO'}")

print("\nResultados carregados com sucesso!")

In [0]:
# ============================================================
# SECAO 4 - DATASET PARA TREINAMENTO
# ============================================================
print("=" * 60)
print("SECAO 4 - Carregando dataset para recriacao dos modelos")
print("=" * 60)

# Carregar ml_train
df_train = spark.table(TRAIN_TABLE).toPandas()
print(f"ml_train: {df_train.shape[0]:,} registros, {df_train.shape[1]} colunas")

# Separar features (X) e target (y)
# Excluir SK_ID_CURR (identificador) e TARGET (variavel resposta)
exclude_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X = df_train[feature_cols].copy()
y = df_train['TARGET'].copy()
sk_ids = df_train['SK_ID_CURR'].copy()

print(f"X (features): {X.shape}")
print(f"y (target): {y.shape}")
print(f"SK_ID_CURR mantido apenas para rastreabilidade (NAO usado como feature)")

# Metadados do dataset
training_rows = len(X)
feature_count = X.shape[1]
positive_class_count = int(y.sum())
negative_class_count = int(len(y) - y.sum())
positive_class_rate = positive_class_count / training_rows

print(f"\n--- Metadados do Dataset ---")
print(f"training_rows: {training_rows:,}")
print(f"feature_count: {feature_count}")
print(f"positive_class_count: {positive_class_count:,}")
print(f"negative_class_count: {negative_class_count:,}")
print(f"positive_class_rate: {positive_class_rate:.4f} ({positive_class_rate*100:.2f}%)")

In [0]:
# ============================================================
# SECAO 5-6 - DEFINIR E RECRIAR MODELOS (MESMA CONFIG DO 16)
# ============================================================
print("=" * 60)
print("SECAO 5-6 - Definindo e recriando modelos (mesma config do notebook 16)")
print("=" * 60)

# Stratified split 80/20 (mesmo do notebook 16)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Calcular scale_pos_weight (mesmo do notebook 16)
n_pos = int(y_tr.sum())
n_neg = int(len(y_tr) - n_pos)
scale_pos_weight = n_neg / n_pos
print(f"Train: {X_tr.shape[0]:,} | Validation: {X_val.shape[0]:,}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# Identificar colunas numericas e categoricas
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
print(f"Numericas: {len(numeric_features)} | Categoricas: {len(categorical_features)}")

# Preprocessor (mesma config do notebook 16)
# Numerico: SimpleImputer(median) + StandardScaler
# Categorico: SimpleImputer(most_frequent) + OneHotEncoder
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Definir modelos com os MESMOS hiperparametros do notebook 16
# NAO realizar tuning - usar exatamente os mesmos parametros
model_configs = {
    'Dummy (most_frequent)': {
        'model': DummyClassifier(strategy='most_frequent', random_state=42),
        'registry_name': 'CreditRisk_Dummy',
        'framework': 'scikit-learn',
    },
    'Dummy (stratified)': {
        'model': DummyClassifier(strategy='stratified', random_state=42),
        'registry_name': 'CreditRisk_DummyStratified',
        'framework': 'scikit-learn',
    },
    'Logistic Regression': {
        'model': LogisticRegression(
            class_weight='balanced', max_iter=2000, solver='lbfgs',
            random_state=42, n_jobs=-1
        ),
        'registry_name': 'CreditRisk_LogisticRegression',
        'framework': 'scikit-learn',
    },
    'Random Forest': {
        'model': RandomForestClassifier(
            n_estimators=200, max_depth=15, min_samples_leaf=50,
            class_weight='balanced', n_jobs=-1, random_state=42
        ),
        'registry_name': 'CreditRisk_RandomForest',
        'framework': 'scikit-learn',
    },
}

if xgb_available:
    model_configs['XGBoost'] = {
        'model': XGBClassifier(
            scale_pos_weight=scale_pos_weight, n_estimators=200, max_depth=6,
            learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, eval_metric='logloss'
        ),
        'registry_name': 'CreditRisk_XGBoost',
        'framework': 'xgboost',
    }

if lgb_available:
    model_configs['LightGBM'] = {
        'model': LGBMClassifier(
            scale_pos_weight=scale_pos_weight, n_estimators=200, max_depth=6,
            learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbose=-1
        ),
        'registry_name': 'CreditRisk_LightGBM',
        'framework': 'lightgbm',
    }

# Treinar modelos (recriar pois nao foram persistidos como artefatos)
# model_recreated = True pois os objetos do notebook 16 nao estao disponiveis
trained_pipelines = {}
model_recreated = True

print(f"\nModelos para recriar: {list(model_configs.keys())}")
for model_name, config in model_configs.items():
    print(f"\nTreinando {model_name}...")
    t0 = time.time()
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', config['model'])
    ])
    pipeline.fit(X_tr, y_tr)
    train_time = time.time() - t0
    trained_pipelines[model_name] = pipeline
    print(f"  Tempo: {train_time:.1f}s | model_recreated={model_recreated}")

print(f"\nTotal de modelos treinados: {len(trained_pipelines)}")

In [0]:
# ============================================================
# SECOES 7-13 - LOGGING INDIVIDUAL DOS MODELOS NO MLFLOW
# ============================================================
print("=" * 60)
print("SECOES 7-13 - Logging individual dos modelos no MLflow")
print("=" * 60)

from mlflow.models import infer_signature

# Tags padronizadas (comuns a todos os modelos)
common_tags = {
    'project': 'Credit Risk Intelligence Platform',
    'domain': 'Credit Risk',
    'dataset': 'Home Credit Default Risk',
    'layer': 'Gold',
    'stage': 'evaluation',
    'random_state': '42',
    'validation_strategy': 'holdout_20_percent',
    'target': 'TARGET',
    'execution_id': EXECUTION_ID,
    'notebook': '18_mlflow',
}

# Dicionario para armazenar info de cada run
mlflow_runs = {}

# Amostra para input_example e signature (5 linhas reais de X)
sample_X = X_val.head(5)

for model_name, config in model_configs.items():
    print(f"\n--- Processando: {model_name} ---")
    run_id = None
    try:
        with mlflow.start_run(run_name=model_name) as run:
            run_id = run.info.run_id
            pipeline = trained_pipelines[model_name]
            classifier = config['model']

            # --- PARAMETROS (Secao 7) ---
            mlflow.log_param('model_type', model_name)
            mlflow.log_param('random_state', 42)
            mlflow.log_param('feature_count', feature_count)
            mlflow.log_param('training_rows', training_rows)
            mlflow.log_param('positive_class_rate', round(positive_class_rate, 6))
            mlflow.log_param('model_recreated', True)
            mlflow.log_param('default_threshold', 0.50)

            # Log todos os hiperparametros do modelo (nao inventar)
            params = classifier.get_params()
            for param_name, param_value in params.items():
                if param_value is not None:
                    mlflow.log_param(param_name, str(param_value))

            # --- METRICAS (Secao 8) ---
            # Metricas de treinamento (do notebook 16)
            train_row = df_training_results[df_training_results['model_name'] == model_name]
            if len(train_row) > 0:
                row = train_row.iloc[0]
                mlflow.log_metric('training_roc_auc', float(row['roc_auc']))
                mlflow.log_metric('training_pr_auc', float(row['pr_auc']))
                mlflow.log_metric('training_recall', float(row['recall']))
                mlflow.log_metric('training_precision', float(row['precision']))
                mlflow.log_metric('training_f1', float(row['f1']))
                mlflow.log_metric('training_accuracy', float(row['accuracy']))
                if pd.notna(row.get('cv_mean')):
                    mlflow.log_metric('cv_mean_roc_auc', float(row['cv_mean']))
                if pd.notna(row.get('cv_std')):
                    mlflow.log_metric('cv_std_roc_auc', float(row['cv_std']))

            # Metricas de validacao (do notebook 17)
            eval_row = df_eval_results[df_eval_results['MODEL'] == model_name]
            if len(eval_row) > 0:
                row = eval_row.iloc[0]
                mlflow.log_metric('validation_roc_auc', float(row['ROC_AUC']))
                mlflow.log_metric('validation_pr_auc', float(row['PR_AUC']))
                mlflow.log_metric('validation_precision', float(row['PRECISION']))
                mlflow.log_metric('validation_recall', float(row['RECALL']))
                mlflow.log_metric('validation_f1', float(row['F1']))
                mlflow.log_metric('validation_specificity', float(row['SPECIFICITY']))
                mlflow.log_metric('validation_balanced_accuracy', float(row['BALANCED_ACCURACY']))
                mlflow.log_metric('validation_brier_score', float(row['BRIER_SCORE']))
                mlflow.log_metric('validation_accuracy', float(row['ACCURACY']))

            # --- TAGS (Secao 10) ---
            model_tags = common_tags.copy()
            model_tags['framework'] = config['framework']
            model_tags['model_type'] = model_name
            for tag_key, tag_value in model_tags.items():
                mlflow.set_tag(tag_key, tag_value)

            # --- ARTEFATOS (Secao 11) ---
            artifact_dir = tempfile.mkdtemp(prefix=f'mlflow_{model_name.replace(" ", "_")}_')

            # model_summary.csv
            summary_df = pd.DataFrame([{
                'model_name': model_name,
                'framework': config['framework'],
                'training_rows': training_rows,
                'validation_rows': X_val.shape[0],
                'feature_count': feature_count,
                'random_state': 42,
                'default_threshold': 0.50,
            }])
            summary_df.to_csv(os.path.join(artifact_dir, 'model_summary.csv'), index=False)

            # evaluation_metrics.csv
            if len(eval_row) > 0:
                eval_metrics_df = eval_row[['MODEL', 'ROC_AUC', 'PR_AUC', 'PRECISION', 'RECALL', 'F1', 'SPECIFICITY', 'BALANCED_ACCURACY', 'BRIER_SCORE']].copy()
            else:
                eval_metrics_df = pd.DataFrame()
            eval_metrics_df.to_csv(os.path.join(artifact_dir, 'evaluation_metrics.csv'), index=False)

            # threshold_analysis.csv (filtrado para este modelo)
            threshold_model = df_threshold[df_threshold['MODEL'] == model_name].copy()
            threshold_model.to_csv(os.path.join(artifact_dir, 'threshold_analysis.csv'), index=False)

            # feature_importance.csv (filtrado para este modelo)
            fi_model = df_feature_importance[df_feature_importance['model_name'] == model_name].copy()
            if len(fi_model) > 0:
                fi_model.to_csv(os.path.join(artifact_dir, 'feature_importance.csv'), index=False)
                # feature_importance_top20.csv (para modelos de arvore)
                fi_top20 = fi_model.nsmallest(20, 'ranking')
                fi_top20.to_csv(os.path.join(artifact_dir, 'feature_importance_top20.csv'), index=False)

            mlflow.log_artifacts(artifact_dir)

            # --- SIGNATURE E INPUT EXAMPLE (Secao 13) ---
            sample_proba = pipeline.predict_proba(sample_X)[:, 1]
            signature = infer_signature(sample_X, sample_proba)

            # --- LOG DO MODELO (Secao 12) ---
            # Todos os modelos sao sklearn Pipelines (preprocessor + classifier).
            # XGBoost e LightGBM estao encapsulados em Pipeline sklearn,
            # portanto mlflow.sklearn.log_model e o flavor correto para todos.
            # mlflow.xgboost.log_model e mlflow.lightgbm.log_model requerem
            # o modelo raw (sem Pipeline), o que nao se aplica aqui.
            log_kwargs = {
                LOG_MODEL_KWARG: 'model',
                'signature': signature,
                'input_example': sample_X,
            }
            model_info = mlflow.sklearn.log_model(pipeline, **log_kwargs)

            mlflow_runs[model_name] = {
                'run_id': run_id,
                'model_uri': model_info.model_uri,
                'registry_name': config['registry_name'],
                'framework': config['framework'],
                'model_recreated': True,
            }

            print(f"  Run ID: {run_id}")
            print(f"  Model URI: {model_info.model_uri}")
            print(f"  Parametros, metricas, tags e artefatos registrados")
            print(f"  Modelo logado com sklearn flavor (Pipeline sklearn)")

    except Exception as e:
        print(f"  ERRO: {model_name} - {str(e)}")
        mlflow_runs[model_name] = {
            'run_id': run_id,
            'model_uri': None,
            'registry_name': config['registry_name'],
            'framework': config['framework'],
            'model_recreated': True,
            'error': str(e),
        }

runs_ok = len([v for v in mlflow_runs.values() if v.get('model_uri')])
print(f"\nTotal de runs MLflow criados com sucesso: {runs_ok}")

In [0]:
# ============================================================
# SECAO 14-15 - MODEL REGISTRY (Unity Catalog)
# ============================================================
print("=" * 60)
print("SECAO 14-15 - Model Registry (Unity Catalog)")
print("=" * 60)

from mlflow import MlflowClient

registry_client = MlflowClient(registry_uri="databricks-uc")

# Metadados de cada modelo registrado (Secao 15)
registry_metadata = []

for model_name, run_info in mlflow_runs.items():
    if run_info.get('model_uri') is None:
        print(f"\n--- {model_name}: PULADO (sem model_uri) ---")
        registry_metadata.append({
            'model_name': f"{REGISTRY_CATALOG}.{REGISTRY_SCHEMA}.{run_info['registry_name']}",
            'model_type': model_name,
            'run_id': run_info.get('run_id'),
            'model_uri': None,
            'status': 'FAILED',
            'error': run_info.get('error', 'No model_uri'),
        })
        continue

    registry_name_full = f"{REGISTRY_CATALOG}.{REGISTRY_SCHEMA}.{run_info['registry_name']}"
    print(f"\n--- Registrando: {model_name} ---")
    print(f"  Registry name: {registry_name_full}")

    try:
        # Registrar modelo no Unity Catalog (cria nova versao a cada execucao)
        registered_version = mlflow.register_model(
            model_uri=run_info['model_uri'],
            name=registry_name_full,
            await_registration_for=300,
        )
        model_version = registered_version.version
        print(f"  Version: {model_version}")

        # Buscar metricas de validacao para metadados
        eval_row = df_eval_results[df_eval_results['MODEL'] == model_name]
        metrics = {}
        if len(eval_row) > 0:
            row = eval_row.iloc[0]
            metrics = {
                'roc_auc': float(row['ROC_AUC']),
                'pr_auc': float(row['PR_AUC']),
                'precision': float(row['PRECISION']),
                'recall': float(row['RECALL']),
                'f1': float(row['F1']),
                'specificity': float(row['SPECIFICITY']),
                'balanced_accuracy': float(row['BALANCED_ACCURACY']),
                'brier_score': float(row['BRIER_SCORE']),
            }

        meta = {
            'model_name': registry_name_full,
            'model_type': model_name,
            'run_id': run_info['run_id'],
            'experiment_id': experiment.experiment_id,
            'model_version': model_version,
            'model_uri': f"models:/{registry_name_full}/{model_version}",
            'training_rows': training_rows,
            'validation_rows': X_val.shape[0],
            'feature_count': feature_count,
            'training_timestamp': START_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S'),
            'status': 'SUCCESS',
        }
        meta.update(metrics)
        registry_metadata.append(meta)

        print(f"  model_uri: models:/{registry_name_full}/{model_version}")
        print(f"  Registrado com SUCESSO")

    except Exception as e:
        print(f"  ERRO ao registrar: {str(e)}")
        registry_metadata.append({
            'model_name': registry_name_full,
            'model_type': model_name,
            'run_id': run_info.get('run_id'),
            'model_uri': None,
            'status': 'FAILED',
            'error': str(e),
        })

models_registered = len([m for m in registry_metadata if m.get('status') == 'SUCCESS'])
print(f"\nTotal de modelos registrados: {models_registered}")

In [0]:
# ============================================================
# SECAO 16 - TABELA DE REGISTRO DE MODELOS
# ============================================================
print("=" * 60)
print("SECAO 16 - Criando tabela credit_risk.analytics.mlflow_model_registry")
print("=" * 60)

# Preparar DataFrame para a tabela (append-only)
registry_records = []
for meta in registry_metadata:
    registry_records.append({
        'execution_id': EXECUTION_ID,
        'model_name': meta.get('model_name'),
        'model_type': meta.get('model_type'),
        'run_id': meta.get('run_id'),
        'experiment_id': meta.get('experiment_id'),
        'model_version': meta.get('model_version'),
        'model_uri': meta.get('model_uri'),
        'training_rows': meta.get('training_rows'),
        'validation_rows': meta.get('validation_rows'),
        'feature_count': meta.get('feature_count'),
        'roc_auc': meta.get('roc_auc'),
        'pr_auc': meta.get('pr_auc'),
        'precision': meta.get('precision'),
        'recall': meta.get('recall'),
        'f1': meta.get('f1'),
        'specificity': meta.get('specificity'),
        'balanced_accuracy': meta.get('balanced_accuracy'),
        'brier_score': meta.get('brier_score'),
        'status': meta.get('status', 'UNKNOWN'),
        'created_at': START_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S'),
    })

df_registry = pd.DataFrame(registry_records)

# Criar tabela Delta (append-only, nao apaga historico)
spark.createDataFrame(df_registry).write.format("delta").mode("append").saveAsTable(
    "credit_risk.analytics.mlflow_model_registry"
)

print(f"Registros inseridos: {len(df_registry)}")
display(df_registry[['model_name', 'model_type', 'model_version', 'roc_auc', 'pr_auc', 'status']])

In [0]:
# ============================================================
# SECAO 17 - AUDITORIA
# ============================================================
print("=" * 60)
print("SECAO 17 - Criando tabela credit_risk.analytics.audit_mlflow")
print("=" * 60)

execution_time = time.time() - START_TIME
models_processed = len(mlflow_runs)
models_registered = len([m for m in registry_metadata if m.get('status') == 'SUCCESS'])
models_failed = len([m for m in registry_metadata if m.get('status') == 'FAILED'])

if models_failed > 0:
    status = 'WARNING' if models_registered > 0 else 'FAIL'
    notes = f"{models_registered} registrados, {models_failed} falharam"
else:
    status = 'SUCCESS'
    notes = f"Todos os {models_registered} modelos registrados com sucesso"

audit_record = pd.DataFrame([{
    'execution_id': EXECUTION_ID,
    'execution_timestamp': START_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S'),
    'experiment_name': EXPERIMENT_NAME,
    'models_processed': models_processed,
    'models_registered': models_registered,
    'models_failed': models_failed,
    'status': status,
    'execution_time_seconds': round(execution_time, 1),
    'notes': notes,
}])

# Append-only (nao sobrescreve historico)
spark.createDataFrame(audit_record).write.format("delta").mode("append").saveAsTable(
    "credit_risk.analytics.audit_mlflow"
)

print(f"Status: {status}")
print(f"Models processed: {models_processed}")
print(f"Models registered: {models_registered}")
print(f"Models failed: {models_failed}")
print(f"Execution time: {execution_time:.1f}s")

In [0]:
# ============================================================
# SECAO 18 - VALIDACAO DOS MODELOS REGISTRADOS
# ============================================================
print("=" * 60)
print("SECAO 18 - Validacao dos modelos registrados")
print("=" * 60)

validation_results = []
sample_validation = X_val.head(10)

for meta in registry_metadata:
    model_name = meta.get('model_type')
    model_uri = meta.get('model_uri')

    if model_uri is None or meta.get('status') != 'SUCCESS':
        print(f"\n--- {model_name}: PULADO (nao registrado) ---")
        validation_results.append({
            'MODEL': model_name,
            'MODEL_URI': model_uri,
            'MODEL_LOAD_TEST': 'SKIP',
            'PREDICTION_TEST': 'SKIP',
            'SIGNATURE_TEST': 'SKIP',
            'STATUS': 'FAIL',
        })
        continue

    print(f"\n--- Validando: {model_name} ---")
    print(f"  URI: {model_uri}")

    try:
        # 1. Carregar modelo utilizando model_uri
        loaded_model = mlflow.pyfunc.load_model(model_uri)
        print(f"  MODEL_LOAD_TEST: PASS")
        load_test = 'PASS'

        # 2. Fazer pequena inferencia (amostra de validacao)
        predictions = loaded_model.predict(sample_validation)
        has_predictions = predictions is not None and len(predictions) > 0
        pred_test = 'PASS' if has_predictions else 'FAIL'
        print(f"  PREDICTION_TEST: {pred_test} (predictions: {len(predictions) if has_predictions else 0})")

        # 3. Validar assinatura
        signature = loaded_model.metadata.signature
        sig_test = 'PASS' if signature is not None else 'WARNING'
        print(f"  SIGNATURE_TEST: {sig_test}")

        # 4. Validar quantidade de features
        if signature and signature.inputs:
            n_features_sig = len(signature.inputs)
            feature_test = 'PASS' if n_features_sig == feature_count else 'WARNING'
            print(f"  Features na assinatura: {n_features_sig} (esperado: {feature_count}) - {feature_test}")
        else:
            feature_test = 'WARNING'

        # 5. Status geral
        overall_status = 'PASS' if all(t == 'PASS' for t in [load_test, pred_test, sig_test]) else 'WARNING'

        validation_results.append({
            'MODEL': model_name,
            'MODEL_URI': model_uri,
            'MODEL_LOAD_TEST': load_test,
            'PREDICTION_TEST': pred_test,
            'SIGNATURE_TEST': sig_test,
            'STATUS': overall_status,
        })

    except Exception as e:
        print(f"  ERRO: {str(e)}")
        validation_results.append({
            'MODEL': model_name,
            'MODEL_URI': model_uri,
            'MODEL_LOAD_TEST': 'FAIL',
            'PREDICTION_TEST': 'FAIL',
            'SIGNATURE_TEST': 'FAIL',
            'STATUS': 'FAIL',
        })

df_validation = pd.DataFrame(validation_results)
print(f"\n--- Resumo da Validacao ---")
display(df_validation)

In [0]:
# ============================================================
# SECAO 19 - RESUMO FINAL
# ============================================================
print("=" * 60)
print("SECAO 19 - Resumo Final")
print("=" * 60)

# Compilar tabela de resumo (nao ordenar por "melhor")
summary_records = []
for meta in registry_metadata:
    summary_records.append({
        'MODEL': meta.get('model_type'),
        'RUN_ID': meta.get('run_id'),
        'MODEL_VERSION': meta.get('model_version'),
        'ROC_AUC': meta.get('roc_auc'),
        'PR_AUC': meta.get('pr_auc'),
        'PRECISION': meta.get('precision'),
        'RECALL': meta.get('recall'),
        'F1': meta.get('f1'),
        'BRIER': meta.get('brier_score'),
        'STATUS': meta.get('status', 'UNKNOWN'),
    })

df_summary = pd.DataFrame(summary_records)
print("\n=== Resumo dos Modelos Registrados ===")
display(df_summary)

In [0]:
# ============================================================
# SECAO 20 - RESUMO EXECUTIVO
# ============================================================
execution_time_final = time.time() - START_TIME
all_pass = all(v['STATUS'] == 'PASS' for v in validation_results)
registry_success = models_registered == models_processed

if registry_success and all_pass:
    overall_status = 'SUCCESS'
elif models_registered > 0:
    overall_status = 'WARNING'
else:
    overall_status = 'FAIL'

print("=" * 50)
print("18_MLFLOW - SUMMARY")
print("=" * 50)
print()
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Experiment ID: {experiment.experiment_id}")
print()
print(f"Models processed: {models_processed}")
print(f"Models registered: {models_registered}")
print()
print(f"Training rows: {training_rows:,}")
print(f"Features: {feature_count}")
print()
print("Registered models:")
for meta in registry_metadata:
    if meta.get('status') == 'SUCCESS':
        print(f"  - {meta['model_name']} (version {meta['model_version']})")
print()
print("Validation metrics: Loaded from 17_evaluation")
print()
print(f"Model Registry: {'SUCCESS' if registry_success else 'WARNING' if models_registered > 0 else 'FAIL'}")
print()
print("Model load tests:")
for v in validation_results:
    print(f"  {v['MODEL']}: {v['STATUS']}")
print()
print("Audit table: credit_risk.analytics.audit_mlflow")
print("Registry table: credit_risk.analytics.mlflow_model_registry")
print()
print(f"Execution time: {execution_time_final:.1f}s ({execution_time_final/60:.1f} min)")
print()
print(f"Status: {overall_status}")
print("=" * 50)